# Phala TDX smoke test

End-to-end check of key derivation, quote binding, mrtd extraction via dcap-qvl and HPKE on real Intel TDX hardware

In [ ]:
!pip install -q dstack-sdk
!pip install -q --force-reinstall "cryptography<47" pyhpke dcap-qvl

import hashlib
from dstack_sdk import DstackClient

c = DstackClient()
info = c.info()
print("app_id:", info.app_id)
print("mrtd  :", info.tcb_info.mrtd)

## Quote parsing with dcap-qvl

In [ ]:
import dcap_qvl


def parse_tdx_quote(raw: bytes):
    quote = dcap_qvl.parse_quote(raw)
    if hasattr(quote, "is_tdx") and not quote.is_tdx():
        raise RuntimeError("not a TDX quote")
    report = quote.report
    mr_td = getattr(report, "mr_td", None)
    if mr_td is None:
        mr_td = getattr(report, "mrtd", None)
    if mr_td is None:
        raise RuntimeError("dcap-qvl report is missing mr_td")
    return bytes(report.report_data), bytes(mr_td)

## Key provider

In [ ]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from pyhpke.keys.x25519_key import X25519Key

KEY_PATH    = "gonka/hpke/x25519/v1"
HKDF_SALT   = b"gonka/hpke/x25519/v1"
HKDF_INFO   = b"gonka/hpke/x25519/v1/key"


def bind_report_data(pk: bytes) -> bytes:
    return hashlib.sha256(pk).digest() + b"\x00" * 32


class DstackKeyProvider:
    def __init__(self, key_path: str = KEY_PATH):
        self._client = DstackClient()
        seed = bytes.fromhex(self._client.get_key(key_path).key)[:32]
        priv_bytes = HKDF(
            algorithm=hashes.SHA256(),
            length=32,
            salt=HKDF_SALT,
            info=HKDF_INFO + b"/" + key_path.encode(),
        ).derive(seed)
        priv = X25519PrivateKey.from_private_bytes(priv_bytes)
        self.sk = X25519Key(priv)
        self.pk = X25519Key(priv.public_key()).to_public_bytes()

        rd = bind_report_data(self.pk)
        self.quote = bytes.fromhex(self._client.get_quote(rd).quote)
        quote_report_data, mr_td = parse_tdx_quote(self.quote)
        if quote_report_data != rd:
            raise RuntimeError("quote report_data does not bind to our pubkey")
        self.mrtd = mr_td.hex()

## Derive key and save artifacts

In [ ]:
p = DstackKeyProvider()
print("pubkey:", p.pk.hex())
print("mrtd  :", p.mrtd)

open("quote.bin",  "wb").write(p.quote)
open("pubkey.bin", "wb").write(p.pk)
print("saved quote.bin / pubkey.bin")

## Sanity checks

In [ ]:
quote_report_data, mr_td = parse_tdx_quote(p.quote)
assert quote_report_data == bind_report_data(p.pk)
assert mr_td.hex() == p.mrtd
assert p.mrtd == info.tcb_info.mrtd
assert DstackKeyProvider().pk == p.pk

print("report_data binding         : ok")
print("mrtd quote == provider      : ok")
print("mrtd provider == dstack.info: ok")
print("derivation determinism      : ok")
print("quote_hash (sha256 of quote):", hashlib.sha256(p.quote).hexdigest())

## HPKE round-trip

In [ ]:
from pyhpke import AEADId, CipherSuite, KDFId, KEMId
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PublicKey

suite = CipherSuite.new(
    KEMId.DHKEM_X25519_HKDF_SHA256,
    KDFId.HKDF_SHA256,
    AEADId.CHACHA20_POLY1305,
)
pkr  = X25519Key(X25519PublicKey.from_public_bytes(p.pk))
info_label = b"gonka/devshard/hpke-info/v1"
aad        = b"gonka/devshard/req/v1"
plaintext  = b'{"hello": "tee"}'

enc, sender = suite.create_sender_context(pkr, info=info_label)
ct = sender.seal(plaintext, aad=aad)
pt = suite.create_recipient_context(enc, p.sk, info=info_label).open(ct, aad=aad)
assert pt == plaintext

print("HPKE round-trip OK")
print("enc       :", enc.hex()[:32], "...")
print("ciphertext:", ct.hex()[:32], "...")
print("plaintext :", pt)

## Next: run Go verifier locally

```bash
cd decentralized-api
go run ./cmd/tdxlite_check \
  ~/Downloads/quote.bin \
  ~/Downloads/pubkey.bin
```

It should print the same MRTD as the `sanity` cell above